# Прослушивание: лабораторная работа №5

Ноутбук для прослушивания артефактов из `labs/lab5_service/audio`. Запускать из каталога `labs/lab5_service`, рядом с папкой `audio`.

Все файлы синтезированы на GTX 1080 Ti: акустическая модель `baseline_gt` из работы №4, вокодер HiFi-GAN, предобученный на LibriTTS.

## Проверка вокодера

Копи-синтез без акустической модели: настоящая запись из корпуса, её мел и два способа озвучивания того же мела. Модель здесь не участвует, поэтому разница показывает только качество вокодера.

- `natural` — настоящая запись;
- `copysynth_hifigan` — тот же мел через HiFi-GAN;
- `copysynth_griffinlim` — тот же мел через Griffin-Lim, так звучала работа №4.

In [ ]:
from IPython.display import Audio, display, Markdown


def play(name, caption=None):
    display(Markdown(f'**{caption or name}** — `audio/{name}.wav`'))
    display(Audio(f'audio/{name}.wav'))


for name in ('natural', 'copysynth_hifigan', 'copysynth_griffinlim'):
    play(name)

## Тестовые фразы

Десять фраз из `data/evaluation_text.csv` через весь тракт: нормализатор, предиктор пауз, транскриптор, акустическая модель, вокодер.

Фразы подобраны под слабые места: омографы `все`/`всё`, иностранные имена, латиница, длинный текст с числами.

In [ ]:
rows = [line.rstrip('\n').split('|', 1)
        for line in open('data/evaluation_text.csv', encoding='utf-8')][1:]
for utt_id, text in rows:
    short = text if len(text) <= 70 else text[:70] + '…'
    play(utt_id, f'{utt_id}: {short}')

## Управление паузами

Пять вариантов одной фразы. Первые три задают паузы знаками препинания, последние два — тегами поверх противоречащей пунктуации.

- `eval_10_3` должен звучать как `eval_10_1`;
- `eval_10_4` должен звучать как `eval_10_2`.

Файлы этих пар совпадают побайтно, поэтому разницы быть не должно.

In [ ]:
variants = [
    ('eval_10', 'без запятой'),
    ('eval_10_1', 'запятая после «казнить»'),
    ('eval_10_2', 'запятая после «нельзя»'),
    ('eval_10_3', 'теги: пауза после «казнить», запрет после «нельзя»'),
    ('eval_10_4', 'теги: запрет после «казнить», пауза после «нельзя»'),
]
for name, caption in variants:
    play(name, f'{name}: {caption}')

## Замеры

Медиана по трём прогонам на фразу, прогрев исключён. RTF — отношение времени синтеза к длительности звука.

In [ ]:
import pandas as pd

df = pd.read_csv('audio/timings.csv', sep='|')
df[['id', 'chars', 'audio_s', 'total_ms', 'rtf', 'pauses_ms', 'acoustic_ms', 'vocoder_ms']]

## Что слышно

- речь разборчивая, но роботизированная: в акустической модели нет предикторов высоты тона и энергии;
- путаются `ё` и `е`, поэтому омографы вроде `все`/`всё` звучат не в том варианте;
- латиница читается по буквам русскими фонемами;
- числа не произносятся: в `1800-х` от токена остаётся только «х»;
- копи-синтез через HiFi-GAN близок к оригиналу, Griffin-Lim на том же меле грубее.